In [5]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

sold = pd.read_csv(
    PROCESSED_DIR / "crmls_sold_features_202401_202606.csv", low_memory=False
)
print(f"{sold.shape[0]:,} rows x {sold.shape[1]} columns")

pd.set_option("display.max_rows", None)
display(pd.DataFrame({"column": sold.columns}))


447,953 rows x 75 columns


,column
0,Flooring
1,ViewYN
2,PoolPrivateYN
3,OriginalListPrice
4,ListingKey
5,CloseDate
6,ClosePrice
7,Latitude
8,Longitude
9,UnparsedAddress


In [6]:
def iqr_bounds(series, k=1.5):
    """Standard IQR fences: [Q1 - k*IQR, Q3 + k*IQR]."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

for col in ["ClosePrice", "LivingArea", "DaysOnMarket"]:
    s = sold[col].dropna()
    lower, upper = iqr_bounds(s)
    n_out = ((s < lower) | (s > upper)).sum()
    print(f"\n{col}:")
    print(f"  Q1={s.quantile(0.25):,.0f}  median={s.median():,.0f}  Q3={s.quantile(0.75):,.0f}")
    print(f"  IQR fences: [{lower:,.0f}, {upper:,.0f}]")
    print(f"  flagged: {n_out:,} ({n_out/len(s)*100:.2f}%)  |  99th pct = {s.quantile(0.99):,.0f}")



ClosePrice:
  Q1=575,000  median=825,000  Q3=1,300,000
  IQR fences: [-512,500, 2,387,500]
  flagged: 33,536 (7.49%)  |  99th pct = 5,600,000

LivingArea:
  Q1=1,248  median=1,646  Q3=2,224
  IQR fences: [-216, 3,688]
  flagged: 19,566 (4.37%)  |  99th pct = 5,288

DaysOnMarket:
  Q1=8  median=18  Q3=48
  IQR fences: [-52, 108]
  flagged: 34,159 (7.63%)  |  99th pct = 233


In [7]:
for col in ["ClosePrice", "LivingArea", "DaysOnMarket"]:
    s = sold[col].dropna()
    print(f"\n{col}:")
    for k in [1.5, 3.0]:
        lower, upper = iqr_bounds(s, k=k)
        n_out = ((s < lower) | (s > upper)).sum()
        print(f"  k={k}: fences [{max(lower,0):,.0f}, {upper:,.0f}]  "
              f"flagged {n_out:,} ({n_out/len(s)*100:.2f}%)")



ClosePrice:
  k=1.5: fences [0, 2,387,500]  flagged 33,536 (7.49%)
  k=3.0: fences [0, 3,475,000]  flagged 14,460 (3.23%)

LivingArea:
  k=1.5: fences [0, 3,688]  flagged 19,566 (4.37%)
  k=3.0: fences [0, 5,152]  flagged 4,930 (1.10%)

DaysOnMarket:
  k=1.5: fences [0, 108]  flagged 34,159 (7.63%)
  k=3.0: fences [0, 168]  flagged 11,910 (2.66%)


### Choosing the IQR Multiplier: k=1.5 vs k=3.0

Standard IQR fences (k=1.5) flag too much of this right-skewed data. Because
price, size, and days-on-market all have long right tails, the 1.5 multiplier
pushes the upper fence too low and misclassifies normal high-end homes as
outliers. Using k=3.0 (the "extreme outlier" standard) gives far more
sensible cut-offs:

| Field | k=1.5 upper / flagged | k=3.0 upper / flagged |
| --- | --- | --- |
| ClosePrice | \$2.39M → 7.49% | **\$3.48M → 3.23%** |
| LivingArea | 3,688 sqft → 4.37% | **5,152 sqft → 1.10%** |
| DaysOnMarket | 108 days → 7.63% | **168 days → 2.66%** |

**Rationale for k=3.0:**

- **ClosePrice > \$3.48M** genuinely enters the luxury tier, whereas the
  \$2.39M cut-off would flag normal move-up homes across Orange County and the
  Bay Area (where county median prices already reach \$1.2M–\$1.7M).
- **LivingArea > 5,152 sqft** is a genuinely large estate; the median home is
  only 1,646 sqft, so this threshold isolates true size outliers.
- **DaysOnMarket > 168 days** (about 5.5 months) is a genuinely slow sale, not
  a routine one.

Statistically, 1.5×IQR marks "mild" outliers and 3.0×IQR marks "extreme"
outliers, so k=3.0 is a principled choice — not an arbitrary loosening — and
it complements (rather than duplicates) the 99th-percentile flags added in
Week 4-5.


In [8]:
# --- Add IQR outlier flags (k=3.0) for the three key numeric fields. ---
IQR_K = 3.0
iqr_fields = ["ClosePrice", "LivingArea", "DaysOnMarket"]

thresholds = {}
for col in iqr_fields:
    lower, upper = iqr_bounds(sold[col].dropna(), k=IQR_K)
    lower = max(lower, 0)  # a negative price/size/DOM lower fence is meaningless
    flag = f"{col.lower()}_iqr_outlier_flag"
    sold[flag] = ((sold[col] < lower) | (sold[col] > upper)).fillna(False)
    thresholds[col] = (lower, upper)
    print(f"{flag}: {int(sold[flag].sum()):,} rows "
          f"({sold[flag].mean()*100:.2f}%)  fences [{lower:,.0f}, {upper:,.0f}]")

# Combined flag: a sale is atypical if ANY of the three fields is an outlier.
iqr_flag_cols = [f"{c.lower()}_iqr_outlier_flag" for c in iqr_fields]
sold["iqr_outlier_any_flag"] = sold[iqr_flag_cols].any(axis=1)
print(f"\niqr_outlier_any_flag (union): {int(sold['iqr_outlier_any_flag'].sum()):,} rows "
      f"({sold['iqr_outlier_any_flag'].mean()*100:.2f}%)")


closeprice_iqr_outlier_flag: 14,460 rows (3.23%)  fences [0, 3,475,000]
livingarea_iqr_outlier_flag: 4,930 rows (1.10%)  fences [0, 5,152]
daysonmarket_iqr_outlier_flag: 11,910 rows (2.66%)  fences [0, 168]

iqr_outlier_any_flag (union): 26,673 rows (5.95%)


In [9]:
# --- Save two datasets + before/after comparison (Week 7 deliverable). ---
REPORT_DIR = PROJECT_ROOT / "data" / "reports" / "week7_outlier_detection"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Full flagged dataset: every row kept, IQR flags added.
FLAGGED_FILE = PROCESSED_DIR / "crmls_sold_flagged_202401_202606.csv"
sold.to_csv(FLAGGED_FILE, index=False)

# 2. Clean filtered dataset: rows flagged on ANY field removed.
filtered = sold.loc[~sold["iqr_outlier_any_flag"]].copy()
FILTERED_FILE = PROCESSED_DIR / "crmls_sold_filtered_202401_202606.csv"
filtered.to_csv(FILTERED_FILE, index=False)

# 3. Before/after comparison: size and medians.
compare = pd.DataFrame({
    "full_flagged": [
        len(sold),
        sold["ClosePrice"].median(),
        sold["LivingArea"].median(),
        sold["DaysOnMarket"].median(),
        sold["price_per_sqft"].median(),
    ],
    "clean_filtered": [
        len(filtered),
        filtered["ClosePrice"].median(),
        filtered["LivingArea"].median(),
        filtered["DaysOnMarket"].median(),
        filtered["price_per_sqft"].median(),
    ],
}, index=["rows", "median_close_price", "median_living_area",
          "median_dom", "median_price_per_sqft"])
compare["pct_change"] = (
    (compare["clean_filtered"] - compare["full_flagged"])
    / compare["full_flagged"] * 100
).round(2)

print(f"Rows removed: {len(sold) - len(filtered):,} "
      f"({(1 - len(filtered)/len(sold))*100:.2f}%)")
print(f"Saved flagged : {FLAGGED_FILE.name}")
print(f"Saved filtered: {FILTERED_FILE.name}")
display(compare)
compare.to_csv(REPORT_DIR / "before_after_comparison.csv")


Rows removed: 26,673 (5.95%)
Saved flagged : crmls_sold_flagged_202401_202606.csv
Saved filtered: crmls_sold_filtered_202401_202606.csv


,full_flagged,clean_filtered,pct_change
rows,447953.000000,421280.000000,-5.95
median_close_price,825000.000000,808000.000000,-2.06
median_living_area,1646.000000,1614.000000,-1.94
median_dom,18.000000,17.000000,-5.56
median_price_per_sqft,537.303217,530.896432,-1.19
